# LeetCode 3399 — Smallest Substring With Identical Characters II bằng Python

- Link: https://leetcode.com/problems/smallest-substring-with-identical-characters-ii/
- Độ khó: **Hard**
- Chủ đề: **Tìm kiếm nhị phân đáp án**, **chia khối (run)** và đếm số lần lật cần thiết `⌊len / (L+1)⌋`, **trường hợp đặc biệt L = 1** (xâu xen kẽ)

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def minLength(self, s: str, numOps: int) -> int:
```

## 1. Mục tiêu học

1. Đưa bài "tối thiểu hóa khối dài nhất" về câu hỏi có/không: "có làm mọi khối ≤ `L` với ≤ `numOps` lần lật?" — đơn điệu theo `L` ⇒ nhị phân.
2. Chứng minh với `L ≥ 2`: số lần lật tối thiểu là `Σ_run ⌊len / (L + 1)⌋` — các run xử lý **độc lập**.
3. Hiểu vì sao `L = 1` **khác biệt**: lật trong một run có thể làm ký tự trùng với run kề bên ⇒ phải so với hai xâu xen kẽ `0101…`, `1010…`.
4. Viết lời giải `O(n log n)` cho `n ≤ 10^5`.
5. Kiểm chứng bằng vét cạn và bằng DP trạng thái (bản I — LC 3398).

## 2. Đề bài & ví dụ

Xâu nhị phân `s`, được lật tối đa `numOps` ký tự. Tối thiểu hóa độ dài **xâu con dài nhất gồm các ký tự giống nhau**.

**Ví dụ 1:** `"000001", numOps = 1` → `2` (`"001001"`). **Ví dụ 2:** `"0000", numOps = 2` → `1` (`"1010"`). **Ví dụ 3:** `"0101", numOps = 0` → `1`.

**Ràng buộc:** `1 ≤ n ≤ 10^5`, `0 ≤ numOps ≤ n`.

In [1]:
examples = [("000001", 1, 2), ("0000", 2, 1), ("0101", 0, 1)]
for s, k, exp in examples:
    print(repr(s), k, '-> kỳ vọng', exp)

'000001' 1 -> kỳ vọng 2
'0000' 2 -> kỳ vọng 1
'0101' 0 -> kỳ vọng 1


## 3. Kiến thức nền

### 3.1. Run (khối liên tiếp cùng ký tự)

In [2]:
def runs(s):
    out, i = [], 0
    while i < len(s):
        j = i
        while j < len(s) and s[j] == s[i]:
            j += 1
        out.append(j - i)
        i = j
    return out

def longest_run(s):
    return max(runs(s)) if s else 0

print(runs("000001"), runs("0011100"), longest_run("0011100"))

[5, 1] [2, 3, 2] 3


### 3.2. Chia một run dài `len` thành các đoạn ≤ `L` bằng lật

Lật một ký tự trong run tách run thành hai phần (ký tự lật đứng giữa, khác hai bên). Với `t` lần lật ta có tối đa `t + 1` phần, tổng độ dài `len − t` ⇒ cần `len − t ≤ (t + 1)·L` ⇔ `t ≥ (len − L) / (L + 1)` ⇔ `t ≥ ⌊len / (L + 1)⌋`. Đặt các vị trí lật tại `L, 2L+1, 3L+2, …` (mỗi `L + 1` ký tự lật một) đạt đúng cận này.

In [3]:
def flips_for_run(length, L):
    return length // (L + 1)

for length in [5, 6, 7, 11]:
    print(length, [flips_for_run(length, L) for L in range(1, 6)])

5 [2, 1, 1, 1, 0]
6 [3, 2, 1, 1, 1]
7 [3, 2, 1, 1, 1]
11 [5, 3, 2, 2, 1]


### 3.3. Hai xâu xen kẽ

Khối dài nhất bằng 1 ⇔ xâu là `0101…` hoặc `1010…`. Số lần lật = khoảng cách Hamming tới xâu đó.

In [4]:
def alt_cost(s):
    a = sum(1 for i, c in enumerate(s) if c != '01'[i % 2])
    return min(a, len(s) - a)

print(alt_cost("0000"), alt_cost("0101"), alt_cost("000001"))

2 0 2


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Ví dụ 1 `"000001"`, `numOps = 1`: run `[5, 1]`.

- `L = 1`: chi phí xen kẽ `min(3, 3) = 3 > 1` ✗.
- `L = 2`: `⌊5/3⌋ + ⌊1/3⌋ = 1 ≤ 1` ✓ (lật vị trí 2: `001001`).

⇒ đáp án 2.

Vì sao `L = 1` cần riêng: với `"0110"`, run `"11"` theo công thức cần `⌊2/2⌋ = 1` lần lật, nhưng run này chỉ có hai ký tự, **cả hai đều ở biên**; lật ký tự nào cũng tạo ra `"00"` với run `0` kề bên. Với `L ≥ 2`, luôn có thể chọn vị trí lật **không ở biên run** (lùi một vị trí nếu cần), nên các run độc lập.

In [5]:
print('"0110" với L=1 theo công thức run:', sum(r // 2 for r in runs("0110")), '| thực tế (xen kẽ):', alt_cost("0110"))

"0110" với L=1 theo công thức run: 1 | thực tế (xen kẽ): 2


## 5. Cách 1 — Brute Force: thử mọi tập vị trí lật

In [6]:
import itertools

def min_len_brute(s, k):
    n, best = len(s), len(s)
    for r in range(min(k, n) + 1):
        for pos in itertools.combinations(range(n), r):
            t = list(s)
            for p in pos:
                t[p] = '1' if t[p] == '0' else '0'
            best = min(best, longest_run(''.join(t)))
    return best

for s, k, exp in examples:
    print(min_len_brute(s, k), '| kỳ vọng', exp)

2 | kỳ vọng 2
1 | kỳ vọng 1
1 | kỳ vọng 1


**Độ phức tạp:** `O(Σ_{r ≤ k} C(n, r) · n)` — chỉ cho `n ≤ 12`.

## 6. Ý tưởng thuật toán / bước đột phá

1. Nếu `alt_cost(s) ≤ numOps` ⇒ đáp án `1`.
2. Ngược lại, nhị phân `L ∈ [2, n]`: `ok(L) ⇔ Σ_run ⌊len / (L + 1)⌋ ≤ numOps`. Hàm không tăng theo `L` ⇒ tìm `L` nhỏ nhất thỏa.

Mỗi lần kiểm tra `O(số run)` ⇒ tổng `O(n log n)`.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — DP trạng thái cho mỗi `L` + nhị phân (cách của bản I)

`dp[i][c][r]` = số lần lật ít nhất để `s[0..i]` kết thúc bằng ký tự `c` với khối cuối dài `r ≤ L`. `O(n · L)` mỗi lần kiểm tra ⇒ chậm khi `n = 10^5` nhưng là oracle độc lập không dựa vào công thức.

In [7]:
def dp_min_flips(s, L):
    INF = float('inf')
    cur = {(c, 1): (0 if s[0] == c else 1) for c in '01'}
    for ch in s[1:]:
        nxt = {}
        for (c, r), v in cur.items():
            for d in '01':
                cost = v + (ch != d)
                if d == c:
                    if r + 1 > L:
                        continue
                    key = (d, r + 1)
                else:
                    key = (d, 1)
                if cost < nxt.get(key, INF):
                    nxt[key] = cost
        cur = nxt
    return min(cur.values())

def min_len_dp(s, k):
    lo, hi = 1, len(s)
    while lo < hi:
        mid = (lo + hi) // 2
        if dp_min_flips(s, mid) <= k:
            hi = mid
        else:
            lo = mid + 1
    return lo

for s, k, exp in examples:
    print(min_len_dp(s, k), '| kỳ vọng', exp)

2 | kỳ vọng 2
1 | kỳ vọng 1
1 | kỳ vọng 1


### 7.2. Cách 3 — Trường hợp `L = 1` + nhị phân với công thức run (lời giải chính)

In [8]:
def min_len_formula(s, k):
    n = len(s)
    if alt_cost(s) <= k:
        return 1
    R = runs(s)
    lo, hi = 2, n
    while lo < hi:
        mid = (lo + hi) // 2
        if sum(r // (mid + 1) for r in R) <= k:
            hi = mid
        else:
            lo = mid + 1
    return lo

for s, k, exp in examples:
    print(min_len_formula(s, k), '| kỳ vọng', exp)

2 | kỳ vọng 2
1 | kỳ vọng 1
1 | kỳ vọng 1


### 7.3. Cách 4 — Không nhị phân: duyệt `L` tăng dần, dừng sớm

Số lần lật cần với `L` là `Σ ⌊r/(L+1)⌋ ≤ n/(L+1)`. Duyệt `L = 2, 3, …` cho tới khi `≤ numOps`; mỗi bước `O(số run)`. Có thể chậm khi `numOps` nhỏ và đáp án lớn (`O(n · đáp án / …)`), nhưng ta chỉ cần xét các run có độ dài `> L` — sắp run giảm dần và cắt danh sách dần.

In [9]:
def min_len_scan(s, k):
    if alt_cost(s) <= k:
        return 1
    R = sorted(runs(s), reverse=True)
    L = 2
    while True:
        while R and R[-1] <= L:          # run ngắn hơn hoặc bằng L không cần lật nữa
            R.pop()
        if sum(r // (L + 1) for r in R) <= k:
            return L
        L += 1

for s, k, exp in examples:
    print(min_len_scan(s, k), '| kỳ vọng', exp)

2 | kỳ vọng 2
1 | kỳ vọng 1
1 | kỳ vọng 1


## 8. TRACE từng bước

Trace cách 3 trên `"0000000111100"`, `numOps = 2`.

In [10]:
def trace(s, k):
    R = runs(s)
    print('run:', R, '| chi phí xen kẽ:', alt_cost(s))
    if alt_cost(s) <= k:
        print('-> đáp án 1'); return
    lo, hi = 2, len(s)
    while lo < hi:
        mid = (lo + hi) // 2
        need = [r // (mid + 1) for r in R]
        print(f"L = {mid}: lật mỗi run {need} -> tổng {sum(need)} {'<=' if sum(need) <= k else '>'} {k}")
        if sum(need) <= k: hi = mid
        else: lo = mid + 1
    print('đáp án:', lo, '| vét cạn:', min_len_brute(s, k) if len(s) <= 13 else '-')

trace("0000000111100", 2)

run: [7, 4, 2] | chi phí xen kẽ: 6
L = 7: lật mỗi run [0, 0, 0] -> tổng 0 <= 2
L = 4: lật mỗi run [1, 0, 0] -> tổng 1 <= 2
L = 3: lật mỗi run [1, 1, 0] -> tổng 2 <= 2
L = 2: lật mỗi run [2, 1, 0] -> tổng 3 > 2
đáp án: 3 | vét cạn: 3


## 9. Code nộp LeetCode

In [11]:
class Solution:
    def minLength(self, s: str, numOps: int) -> int:
        n = len(s)
        a = sum(1 for i, c in enumerate(s) if c != '01'[i & 1])
        if min(a, n - a) <= numOps:
            return 1
        R = []
        i = 0
        while i < n:
            j = i
            while j < n and s[j] == s[i]:
                j += 1
            R.append(j - i)
            i = j
        lo, hi = 2, n
        while lo < hi:
            mid = (lo + hi) // 2
            need = 0
            for r in R:
                need += r // (mid + 1)
                if need > numOps:
                    break
            if need <= numOps:
                hi = mid
            else:
                lo = mid + 1
        return lo

for s, k, exp in examples:
    got = Solution().minLength(s, k)
    print(got, 'OK' if got == exp else 'SAI')

2 OK
1 OK
1 OK


## 10. Vì sao thuật toán đúng

1. **Cận dưới (mọi `L`):** trong mỗi run gốc độ dài `len`, sau khi lật `t` vị trí **trong run**, các ký tự không lật tạo thành ≤ `t + 1` đoạn liên tiếp cùng ký tự gốc; để mỗi đoạn ≤ `L` cần `t ≥ ⌊len/(L+1)⌋`. Lật ở run khác không giúp tách run này. ⇒ tổng lật ≥ `Σ ⌊len/(L+1)⌋`.
2. **Đạt được khi `L ≥ 2`:** trong run, lật các vị trí `L, 2L+1, …` (chỉ số trong run tính từ 0). Nếu vị trí lật cuối rơi đúng ký tự **cuối** run (khi `len ≡ L (mod L+1)`… cụ thể khi `len` chia hết `L+1`), lùi nó một vị trí: đoạn trước nó dài `L − 1 ≥ 1`, đoạn sau dài 1 — vẫn ≤ `L` và ký tự lật **không** kề run sau. Ký tự lật nằm giữa hai ký tự gốc nên tạo khối độ dài 1. Các run độc lập.
3. **`L = 1`:** xâu phải xen kẽ hoàn toàn ⇒ chi phí đúng bằng `alt_cost`.
4. **Đơn điệu:** làm được với `L` thì làm được với `L + 1` ⇒ nhị phân đúng.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Thử mọi tập lật | mũ | `O(n)` |
| 2. DP + nhị phân | `O(n · L · log n)` (tệ nhất `O(n² log n)`) | `O(L)` |
| 3. Công thức + nhị phân | `O(n log n)` | `O(n)` |
| 4. Duyệt `L` tăng dần | `O(n + Σ_L #run>L)` | `O(n)` |

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [12]:
impls = {'dp': min_len_dp, 'formula': min_len_formula, 'scan': min_len_scan,
         'submit': lambda s, k: Solution().minLength(s, k)}
edge = [("0", 0, 1), ("1", 1, 1), ("00", 0, 2), ("00", 1, 1), ("0011", 1, 2), ("0011", 2, 1), ("000000", 1, 3), ("1111111", 2, 2)]
for s, k, exp in examples + edge:
    assert min_len_brute(s, k) == exp, (s, k, min_len_brute(s, k))
    for name, f in impls.items():
        assert f(s, k) == exp, (name, s, k)
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


### 12.2. Đối chiếu ngẫu nhiên với vét cạn và DP

In [13]:
import random
rng = random.Random(3399)
cnt = 0
for _ in range(1500):
    n = rng.randint(1, 11)
    s = ''.join(rng.choice('01') if rng.random() < 0.5 else '0' for _ in range(n))
    k = rng.randint(0, n)
    exp = min_len_brute(s, k)
    for name, f in impls.items():
        assert f(s, k) == exp, (name, s, k)
    cnt += 1
for _ in range(100):
    n = rng.randint(1, 300)
    s = ''.join(rng.choice('0000001') for _ in range(n)); k = rng.randint(0, n // 3)
    assert min_len_dp(s, k) == Solution().minLength(s, k)
print('Khớp vét cạn trên', cnt, 'bộ nhỏ; khớp DP trên 100 bộ tới 300 ký tự.')

Khớp vét cạn trên 1500 bộ nhỏ; khớp DP trên 100 bộ tới 300 ký tự.


### 12.3. Ca lớn `n = 10^5`

In [14]:
import time
n = 10 ** 5
cases = {
    'toàn 0, k = 1': ('0' * n, 1),
    'toàn 0, k = n/2': ('0' * n, n // 2),
    'run dài ngẫu nhiên, k = 3000': (''.join(rng.choice('0000001') for _ in range(n)), 3000),
    'gần xen kẽ, k = 5': (''.join('01'[i % 2] if i % 997 else '0' for i in range(n)), 5),
}
for name, (s, k) in cases.items():
    t = time.time(); a = Solution().minLength(s, k); t1 = time.time() - t
    t = time.time(); b = min_len_scan(s, k); t2 = time.time() - t
    assert a == b
    print(f"{name:30s}: {a:6d} | nhị phân {t1:.3f}s | duyệt L {t2:.3f}s")

toàn 0, k = 1                 :  50000 | nhị phân 0.007s | duyệt L 0.021s
toàn 0, k = n/2               :      1 | nhị phân 0.006s | duyệt L 0.004s
run dài ngẫu nhiên, k = 3000  :     11 | nhị phân 0.020s | duyệt L 0.014s
gần xen kẽ, k = 5             :      3 | nhị phân 0.060s | duyệt L 0.018s


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Dùng công thức run cho cả `L = 1`

In [15]:
def wrong_no_special(s, k):
    R = runs(s)
    lo, hi = 1, len(s)
    while lo < hi:
        mid = (lo + hi) // 2
        if sum(r // (mid + 1) for r in R) <= k: hi = mid
        else: lo = mid + 1
    return lo

print('sai :', wrong_no_special("0110", 1), '| đúng:', Solution().minLength("0110", 1))

sai : 1 | đúng: 2


**Vì sao sai:** `"0110"` có run `[1, 2, 1]`; công thức nói chỉ cần 1 lần lật (tách run `"11"`), nhưng lật một trong hai số `1` sẽ tạo `"0010"` hoặc `"0100"` — ký tự lật trùng với run kề. Với `L = 1` các run **không** độc lập; thực tế cần 2 lần lật.

### Lỗi 2 — Số lần lật `⌈len / L⌉ − 1` hoặc `len // L`

In [16]:
def make_wrong(f):
    def solve(s, k):
        if alt_cost(s) <= k:
            return 1
        R = runs(s)
        L = 2
        while sum(f(r, L) for r in R) > k:
            L += 1
        return L
    return solve

wrong_div_L = make_wrong(lambda r, L: r // L)
print('sai :', wrong_div_L("000000", 1), '| đúng:', Solution().minLength("000000", 1))

sai : 4 | đúng: 3


**Vì sao sai:** mỗi lần lật "tiêu tốn" một ký tự (ký tự lật không thuộc đoạn nào) ⇒ mỗi lần lật xử lý được `L + 1` ký tự, không phải `L`.

### Lỗi 3 — Chỉ so với một xâu xen kẽ (`0101…`)

In [17]:
def wrong_one_pattern(s, k):
    a = sum(1 for i, c in enumerate(s) if c != '01'[i % 2])
    if a <= k:
        return 1
    return max(2, Solution().minLength(s, k))

print('sai :', wrong_one_pattern("1010", 0), '| đúng:', Solution().minLength("1010", 0))

sai : 2 | đúng: 1


**Vì sao sai:** xâu xen kẽ có thể bắt đầu bằng `1`.

### Lỗi 4 — Tham lam: luôn lật ký tự **giữa** run dài nhất

In [18]:
def wrong_greedy_middle(s, k):
    t = list(s)
    for _ in range(k):
        best_len, best_mid, i = 0, -1, 0
        while i < len(t):
            j = i
            while j < len(t) and t[j] == t[i]:
                j += 1
            if j - i > best_len:
                best_len, best_mid = j - i, (i + j) // 2
            i = j
        if best_len <= 1:
            break
        t[best_mid] = '1' if t[best_mid] == '0' else '0'
    return longest_run(''.join(t))

rng = random.Random(4)
ce = next((s, k) for s, k in ((''.join(rng.choice('001') for _ in range(rng.randint(3, 11))), rng.randint(1, 3)) for _ in range(20000))
          if wrong_greedy_middle(s, k) != min_len_brute(s, k))
print('phản ví dụ:', ce, '| sai:', wrong_greedy_middle(*ce), '| đúng:', min_len_brute(*ce))

phản ví dụ: ('0010', 1) | sai: 2 | đúng: 1


**Vì sao sai:** lật "giữa" một run ngắn có thể đặt ký tự lật **sát run kề** và tạo khối mới (như `"0010"`: lật vị trí 1 cho `"0110"`, trong khi lật vị trí 0 cho `"1010"` — khối dài nhất 1). Hơn nữa, với nhiều lần lật, chia đôi lặp lại không phân bố đều bằng việc lật mỗi `L + 1` ký tự. Phải dùng điều kiện kiểm tra theo `L` (nhị phân), không tham lam cục bộ.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Tối thiểu hóa giá trị lớn nhất" ⇒ nhị phân đáp án.
- Bài trên xâu nhị phân với "khối giống nhau" ⇒ phân tích theo run.
- Mỗi thao tác tách một run và "tốn" một ký tự ⇒ `⌊len/(L+1)⌋`.
- Kiểm tra kỹ trường hợp biên nhỏ nhất (`L = 1`) — tương tác giữa các run.

## 15. Biến thể của bài

1. **LC 3398 — bản I** (`n ≤ 1000`): DP + nhị phân là đủ.
2. Bảng chữ cái lớn hơn 2 ⇒ `L = 1` dễ hơn (luôn có ký tự khác cả hai hàng xóm khi ≥ 3 ký tự), công thức run áp dụng cho mọi `L ≥ 1`.
3. Mỗi thao tác đổi một **đoạn liên tiếp** ⇒ bài khác hẳn.
4. Tối đa hóa khối dài nhất với `k` lần lật ⇒ LC 1004 (cửa sổ trượt).

In [19]:
# Biến thể 2: bảng chữ cái 3 ký tự 'abc', được đổi ký tự tùy ý: số đổi = Σ ⌊len/(L+1)⌋ cho mọi L >= 1
def min_len_ternary(s, k):
    R = runs(s)
    lo, hi = 1, len(s)
    while lo < hi:
        mid = (lo + hi) // 2
        if sum(r // (mid + 1) for r in R) <= k: hi = mid
        else: lo = mid + 1
    return lo

def brute_ternary(s, k):
    n, best = len(s), len(s)
    for r in range(min(k, n) + 1):
        for pos in itertools.combinations(range(n), r):
            for repl in itertools.product('abc', repeat=r):
                t = list(s)
                for p, ch in zip(pos, repl):
                    t[p] = ch
                best = min(best, longest_run(''.join(t)))
    return best

rng = random.Random(15)
for _ in range(150):
    s = ''.join(rng.choice('aab') for _ in range(rng.randint(1, 7))); k = rng.randint(0, 3)
    assert min_len_ternary(s, k) == brute_ternary(s, k)
print('Biến thể 3 ký tự khớp vét cạn: "aaaa", k=1 ->', min_len_ternary("aaaa", 1))

Biến thể 3 ký tự khớp vét cạn: "aaaa", k=1 -> 2


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Ghi chú |
|---|---|---|---|
| 1 | Thử mọi tập lật | mũ | Oracle |
| 2 | DP + nhị phân | `O(n L log n)` | Oracle độc lập |
| 3 | Công thức run + `L = 1` riêng | `O(n log n)` | **Nộp bài** |
| 4 | Duyệt `L` tăng dần | gần tuyến tính | Thay thế |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `s = "111000"`, `numOps = 0`.

**Bài 2 (tính tay):** `s = "0000000000"` (10 ký tự), `numOps = 3`.

**Bài 3 (điền TODO):** viết `min_ops_for(s, L)` — số lần lật ít nhất để mọi khối ≤ `L`.

**Bài 4 (tự viết lại):** dựng **xâu kết quả** cụ thể đạt đáp án.

**Bài 5 (mini challenge):** trả lời nhiều truy vấn `numOps` khác nhau trên cùng `s` trong `O(log n)` mỗi truy vấn sau tiền xử lý.

In [20]:
def min_ops_for_todo(s, L):
    # TODO
    raise NotImplementedError

def build_result_todo(s, numOps):
    # TODO
    raise NotImplementedError

def answer_queries_todo(s, ops_list):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Không lật. (2) Run `[3, 3]`. (3) `3`.

**Bài 2.** (1) Một run dài 10. (2) `L = 2`: `⌊10/3⌋ = 3` ✓. (3) `L = 1`: xen kẽ cần 5 ✗ ⇒ `2`.

**Bài 3.** (1) `L = 1` ⇒ `alt_cost`. (2) `L ≥ 2` ⇒ tổng công thức. (3) Đối chiếu DP.

**Bài 4.** (1) `L = 1` ⇒ trả về xâu xen kẽ gần hơn. (2) `L ≥ 2` ⇒ lật vị trí `L, 2L+1, …` trong mỗi run, lùi vị trí cuối nếu ở biên. (3) Kiểm tra khối dài nhất và số lần lật.

**Bài 5.** (1) Tính `need(L)` cho mọi `L` (không tăng). (2) Với `numOps`, nhị phân `L` nhỏ nhất có `need(L) ≤ numOps`. (3) `need(L)` cho mọi `L` tính trong `O(n log n)` bằng điều hòa (mỗi run đóng góp cho `L < len`).

## 19. Đáp án tham khảo cho bài tập

In [21]:
# Bài 1, 2, 3
assert Solution().minLength("111000", 0) == 3 and Solution().minLength("0" * 10, 3) == 2 == min_len_brute("0" * 10, 3)
def min_ops_for(s, L):
    return alt_cost(s) if L == 1 else sum(r // (L + 1) for r in runs(s))
rng = random.Random(3)
for _ in range(300):
    s = ''.join(rng.choice('001') for _ in range(rng.randint(1, 40))); L = rng.randint(1, 6)
    assert min_ops_for(s, L) == dp_min_flips(s, L)
print('Bài 1: 3 | Bài 2: 2 | Bài 3 khớp DP.')

Bài 1: 3 | Bài 2: 2 | Bài 3 khớp DP.


In [22]:
# Bài 4, 5
def build_result(s, k):
    L = Solution().minLength(s, k)
    n = len(s)
    if L == 1:
        a = ''.join('01'[i % 2] for i in range(n)); b = ''.join('10'[i % 2] for i in range(n))
        return min((sum(x != y for x, y in zip(s, a)), a), (sum(x != y for x, y in zip(s, b)), b))[1]
    t, i = list(s), 0
    while i < n:
        j = i
        while j < n and s[j] == s[i]:
            j += 1
        p = i + L
        while p < j:
            q = p if p != j - 1 else p - 1          # tránh lật ký tự cuối run
            t[q] = '1' if t[q] == '0' else '0'
            p = q + L + 1
        i = j
    return ''.join(t)

def answer_queries(s, ops_list):
    n = len(s)
    R = runs(s)
    need = [0] * (n + 2)
    for r in R:
        for L in range(2, r):                      # chỉ L < r mới cần lật
            need[L] += r // (L + 1)
    need[1] = alt_cost(s)
    out = []
    for k in ops_list:
        lo, hi = 1, n
        while lo < hi:
            mid = (lo + hi) // 2
            if need[mid] <= k: hi = mid
            else: lo = mid + 1
        out.append(lo)
    return out

rng = random.Random(5)
for _ in range(300):
    s = ''.join(rng.choice('0001') for _ in range(rng.randint(1, 40))); k = rng.randint(0, 10)
    t = build_result(s, k)
    assert longest_run(t) == Solution().minLength(s, k) and sum(a != b for a, b in zip(s, t)) <= k
s = ''.join(rng.choice('0001') for _ in range(200))
assert answer_queries(s, list(range(0, 60))) == [Solution().minLength(s, k) for k in range(0, 60)]
print('Bài 4, 5 đúng; ví dụ 1 dựng được:', build_result("000001", 1))

Bài 4, 5 đúng; ví dụ 1 dựng được: 001001


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
if min(alt, n - alt) <= k: return 1            # alt = khoảng cách tới "0101..."
R = độ dài các run
nhị phân L trong [2, n]:  ok(L) <=> Σ r // (L + 1) <= k
```

**Checklist:**

- [ ] `L = 1` xử lý riêng bằng hai xâu xen kẽ?
- [ ] Mỗi lần lật xử lý `L + 1` ký tự (`r // (L+1)`)?
- [ ] So cả `0101…` và `1010…`?
- [ ] Nhị phân tìm `L` **nhỏ nhất** thỏa?
- [ ] Dừng sớm khi tổng vượt `k`?